# Mission 3 — BCE vs Weighted BCE 수정본 v2
기존 실행을 중지하고 새 Colab 노트북에서 처음부터 실행하세요.
이전 실패한 checkpoint는 사용하지 않습니다. 새 출력: /content/m3_loss_control_fixed_v2.

수정: 손실 가중치를 config 숫자로 보관하고 매 forward에서 실제 장치에 생성합니다.
FP32로 두 실험을 수행합니다. attention 점수 clipping은 추가하지 않습니다.
학습 전에 양성/음성 라벨이 포함된 수동 BCE 계산, 작은 모델 저장/재로딩,
실제 데이터의 초기 손실을 검증합니다. 비정상 손실/gradient 및 epoch F1=0은 중단합니다.
검사 통과는 성능 보장이 아니며, B0 결과를 확인한 뒤 8번 L1을 실행하세요.

# Mission 3 — 첫 번째 통제 실험: BCE vs 가중 BCE
새 Colab GPU 노트북에서 셀을 위에서 아래로 실행합니다.
**이번에는 손실함수 한 가지 요인만 변경합니다. 문맥 처리 변경은 다음 실험입니다.**

| 조건 | B0 | L1 |
|---|---|---|
| 모델 | KLUE-RoBERTa Base | 동일 |
| head | 증상별 Attention | 동일 |
| 입력 | 발화 경계 SEP + First-512 | 동일 |
| 손실 | BCE | 가중 BCE |
| threshold | 0.5 | 0.5 |
| 학습 | seed 42, 4 epoch, batch 8, LR 2e-5 | 동일 |
| 결과 선택 | 마지막 epoch | 동일 |

L1 양성 가중치: clip(sqrt(내부 학습 음성 수 / 양성 수), 1, 3).
9개 증상 모두에 같은 계산 규칙을 적용합니다. 검증 점수를 보고 가중치를 탐색하지 않습니다.
이 규칙은 이번 실험에서 미리 정한 가설이지 검증된 최적값은 아닙니다.
주 비교는 RoBERTa 단독 B0 vs L1입니다. TF-IDF는 한 번만 학습하고 양쪽에서 동일하게 재사용합니다.
부 비교로 RoBERTa 0.5 + TF-IDF 0.5 결과도 저장합니다. 두 확률/가중치를 사후 조정하지 않습니다.
B0도 재학습하여 같은 환경에서 비교합니다. 기존 0.6114는 참고값입니다.

준비: Drive Training ZIP, /content/split_manifest.csv, /content/m3_fixed_results.tar.gz.
전체 실행하면 TF-IDF 1회, RoBERTa 2회 학습합니다. 모델 저장은 임시 /content이며 마지막 백업이 필요합니다.
이 실험으로 0.65 달성을 보장하지 않습니다. 내부 검증에서 여러 실험을 선택하므로 독립 테스트가 아닙니다.


In [ ]:
%pip install -q "transformers==5.16.1" "accelerate>=1,<2" "scikit-learn>=1.5,<2" "pandas>=2.2,<3" safetensors "joblib>=1.4,<2"


## 1. 경로·환경
GPU를 선택하세요. 다운로드한 작은 결과 백업과 manifest는 Colab 왼쪽 파일 패널로 /content에 업로드합니다.
원본 모델 압축파일은 필요 없습니다. Drive 경로는 실제 위치에 맞게 수정하세요.


In [ ]:
from pathlib import Path
import json, zipfile, tarfile, hashlib, gc, inspect, platform
import importlib.metadata as im
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import (AutoTokenizer, RobertaModel, RobertaPreTrainedModel,
                          TrainingArguments, Trainer, set_seed, RobertaConfig)
from transformers.modeling_outputs import SequenceClassifierOutput
from google.colab import drive, files
drive.mount("/content/drive")

TRAIN_ZIP = Path("/content/drive/MyDrive/DDC_M3/data/mission3_train_json.zip")
MANIFEST = Path("/content/split_manifest.csv")
OLD_BACKUP = Path("/content/m3_fixed_results.tar.gz")
OUT = Path("/content/m3_loss_control_fixed_v2")
OUT.mkdir(exist_ok=True)
TARGETS = ["고열","구토","두통","복통","어지러움","열상","오심","전신쇠약","호흡곤란"]
MODEL_NAME = "klue/roberta-base"
assert torch.cuda.is_available(), "먼저 GPU 런타임을 선택하세요."
ENV = {"python": platform.python_version(), **{n: im.version(n) for n in
       ["torch","transformers","accelerate","numpy","pandas","scikit-learn"]}}
print(ENV)
print("GPU:", torch.cuda.get_device_name(0))
for p in [TRAIN_ZIP, MANIFEST, OLD_BACKUP]:
    assert p.is_file(), f"파일을 준비하세요: {p}"


## 2. 원본 분할·텍스트 복원
Training만 읽습니다. 공식 Validation은 사용하지 않습니다.
파일 ID는 분할·예측 정렬 검사용이며 모델 입력에 포함하지 않습니다.


In [ ]:
rows = []
with zipfile.ZipFile(TRAIN_ZIP) as z:
    for name in sorted(z.namelist()):
        p = Path(name)
        if "__MACOSX" in p.parts or p.name.startswith("._") or p.suffix.lower() != ".json":
            continue
        d = json.loads(z.read(name))
        texts = [u.get("text", "") for u in d["utterances"]]
        assert all(isinstance(t, str) for t in texts)
        assert isinstance(d["symptom"], list)
        rows.append({"file_name": p.name, "text": " ".join(texts), "utterance_texts": texts,
                     **{t: int(t in d["symptom"]) for t in TARGETS}})
df = pd.DataFrame(rows)
split = pd.read_csv(MANIFEST)
assert len(df) == 29200 and not df.file_name.duplicated().any()
assert not split.file_name.duplicated().any()
assert set(split.file_name) == set(df.file_name)
assert set(split.split) == {"train","dev_valid"}
ordered = split.merge(df, on="file_name", how="left", validate="one_to_one", sort=False)
train = ordered[ordered.split == "train"].reset_index(drop=True)
valid = ordered[ordered.split == "dev_valid"].reset_index(drop=True)
assert (len(train), len(valid)) == (23408, 5792)
split.to_csv(OUT / "split_manifest.csv", index=False)
print("내부 학습:", train.shape, "내부 검증:", valid.shape)
print("manifest SHA256:", hashlib.sha256(MANIFEST.read_bytes()).hexdigest())

# 압축 해제 없이 필요한 예측만 읽습니다.
import io
old = {}
with tarfile.open(OLD_BACKUP, "r:gz") as archive:
    for run in ["tfidf","e0","e1","e3","e0_e3_equal"]:
        member = archive.getmember(f"{run}/dev_predictions.npz")
        with np.load(io.BytesIO(archive.extractfile(member).read()), allow_pickle=False) as a:
            old[run] = {k: a[k].copy() for k in a.files}
        a = old[run]
        assert list(a["targets"]) == TARGETS
        assert np.array_equal(a["file_names"], valid.file_name.to_numpy()), "기존 검증 순서와 다릅니다."
        assert np.array_equal(a["y_true"], valid[TARGETS].to_numpy()), "기존 정답과 다릅니다."
        assert a["probs"].shape == (5792, 9) and np.isfinite(a["probs"]).all()
        assert ((a["probs"] >= 0) & (a["probs"] <= 1)).all()
print("기존 5개 결과와 검증 행 순서·정답 일치 확인 완료")


## 3. 우선순위 1 — 증상별 오류 분석
오심의 낮은 점수가 precision 문제인지 recall 문제인지 확인합니다.
표본을 읽고 근거 누락·부정문·질문·간접 표현·잘린 문맥을 구분하세요.
오심과 오한을 혼동한 규칙을 만들거나 정답을 임의 수정하지 않습니다.
아래 CSV에는 통화 원문이 있으므로 GitHub에 올리지 마세요.


In [ ]:
def label_report(y, probs):
    pred = probs >= .5
    p, r, f, support = precision_recall_fscore_support(y, pred, average=None, zero_division=0)
    return pd.DataFrame({"symptom": TARGETS, "precision": p, "recall": r, "f1": f,
                         "support": support, "predicted_positive": pred.sum(axis=0)})
for name, a in old.items():
    print(name, "Macro F1:", f1_score(a["y_true"], a["probs"] >= .5, average="macro", zero_division=0))
display(label_report(old["e0"]["y_true"], old["e0"]["probs"]).sort_values("f1"))
j = TARGETS.index("오심")
a = old["e0"]
err = valid[["file_name","text"]].copy()
err["truth"] = a["y_true"][:,j]
err["probability"] = a["probs"][:,j]
err["prediction"] = (err.probability >= .5).astype(int)
err = err[err.truth != err.prediction].copy()
err["error_type"] = np.where(err.truth == 1, "FN: 오심 놓침", "FP: 오심 과예측")
err.to_csv(OUT / "e0_nausea_errors_private.csv", index=False, encoding="utf-8-sig")
for kind, group in err.groupby("error_type"):
    print(kind, len(group))
    display(group.sample(min(10, len(group)), random_state=42))


## 4. 고정 입력과 학습 데이터 기반 양성 가중치
두 실험 모두 동일한 발화 경계 입력을 사용합니다.


In [ ]:
import re, unicodedata
MODES = ["raw", "boundary", "boundary_normalized"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def normalize_utterance(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

def encode_call(utterances, mode):
    assert mode in MODES
    if mode == "raw":
        body = tokenizer(" ".join(utterances), add_special_tokens=False, truncation=False)["input_ids"]
    else:
        body = []
        for i, text in enumerate(utterances):
            if i:
                body.append(tokenizer.sep_token_id)
            if mode == "boundary_normalized":
                text = normalize_utterance(text)
            body.extend(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])
    full_length = len(body) + 2
    return [tokenizer.cls_token_id] + body[:510] + [tokenizer.sep_token_id], full_length

class Calls(Dataset):
    def __init__(self, frame, mode):
        encoded = [encode_call(u, mode) for u in frame.utterance_texts]
        self.ids = [x[0] for x in encoded]
        self.lengths = np.array([x[1] for x in encoded])
        self.labels = frame[TARGETS].to_numpy(dtype=np.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "labels": self.labels[i]}

def collate(batch):
    x = tokenizer.pad([{"input_ids": b["input_ids"]} for b in batch], return_tensors="pt")
    x["labels"] = torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float32)
    return x

assert normalize_utterance("토할  것 같아요.\n토는 안 했어요.") == "토할 것 같아요. 토는 안 했어요."
assert normalize_utterance("오심 구토 아니요 없어요") == "오심 구토 아니요 없어요"
for utterances in train.utterance_texts.iloc[:30]:
    actual, _ = encode_call(utterances, "raw")
    expected = tokenizer(" ".join(utterances), truncation=True, max_length=512)["input_ids"]
    assert actual == expected, "P0가 이전 입력과 다릅니다."
for mode in MODES:
    ids_test, _ = encode_call(["", "토는 안 했어요.", "속이 울렁거려요."], mode)
    assert len(ids_test) <= 512 and ids_test[0] == tokenizer.cls_token_id

# 이 단계부터 사용하는 전처리는 boundary 하나뿐입니다.
EXPERIMENTS = ["b0_bce", "l1_weighted_bce"]
train_ds, valid_ds = Calls(train, "boundary"), Calls(valid, "boundary")
y_train = train[TARGETS].to_numpy(dtype=np.float32)
positive = y_train.sum(axis=0)
negative = len(train) - positive
assert (positive > 0).all(), "양성 0개 클래스가 있습니다."
POS_WEIGHTS = np.clip(np.sqrt(negative / positive), 1., 3.).astype(np.float32)
weight_table = pd.DataFrame({"symptom": TARGETS, "positive": positive,
                            "negative": negative, "pos_weight": POS_WEIGHTS})
display(weight_table)
weight_table.to_csv(OUT / "training_pos_weights.csv", index=False)
(OUT / "weight_rule.json").write_text(json.dumps({"rule": "clip(sqrt(negative/positive),1,3)",
    "source": "internal train only", "targets": TARGETS, "weights": POS_WEIGHTS.tolist()}, indent=2))
print("동일 boundary 입력 준비:", len(train_ds), len(valid_ds))


## 5. 동일 모델 + 손실 가중치
두 조건 모두 같은 클래스와 초기화로 시작합니다. 가중치는 config에 저장해 모델 복원 시 유지합니다.


In [ ]:
class SymptomModel(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        h = config.hidden_size
        self.head_type = getattr(config, "head_type", "cls")
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.dense = nn.Linear(h, h)
        self.out_proj = nn.Linear(h, config.num_labels)
        if self.head_type == "label_attention":
            self.attention = nn.Linear(h, config.num_labels, bias=False)
        # Keep loss weights as ordinary config values, not a tensor buffer subject
        # to from_pretrained's meta-device / missing-weight initialization.
        values = getattr(config, "loss_pos_weight", [1.] * config.num_labels)
        if len(values) != config.num_labels or any(not np.isfinite(v) or v <= 0 for v in values):
            raise ValueError("Invalid loss weights")
        config.loss_pos_weight = [float(v) for v in values]
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask,
                              return_dict=True).last_hidden_state
        mask = attention_mask.bool().clone()
        mask[:,0] = False
        last = attention_mask.sum(1).long() - 1
        mask[torch.arange(len(mask), device=mask.device), last] = False
        empty = ~mask.any(dim=1)
        mask[empty,0] = True
        if self.head_type == "cls":
            pooled = hidden[:,0]
        elif self.head_type == "mean":
            pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        else:
            scores = self.attention(hidden).float().masked_fill(~mask.unsqueeze(-1), -1e9)
            weights = scores.softmax(dim=1).to(hidden.dtype)
            pooled = torch.einsum("blc,blh->bch", weights, hidden)
        x = self.dropout(torch.tanh(self.dense(self.dropout(pooled))))
        if self.head_type == "label_attention":
            logits = (x * self.out_proj.weight.unsqueeze(0)).sum(-1) + self.out_proj.bias
        else:
            logits = self.out_proj(x)
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite logits: stop before saving results")
        loss = None
        if labels is not None:
            if labels.shape != logits.shape or not ((labels == 0) | (labels == 1)).all():
                raise ValueError("Labels must be binary and match logits")
            # Construct on the actual device AFTER loading, on each forward pass.
            pos_weight = torch.tensor(self.config.loss_pos_weight, device=logits.device,
                                      dtype=torch.float32)
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits.float(), labels.float(), pos_weight=pos_weight)
            if not torch.isfinite(loss) or loss.detach().item() > 100:
                raise FloatingPointError("Abnormal BCE loss; inspect initialization and labels")
        return SequenceClassifierOutput(loss=loss, logits=logits)

def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    return {"macro_f1": f1_score(p.label_ids, logits >= 0, average="macro", zero_division=0)}

# 다운로드·학습 없는 구조 검사
tiny = RobertaConfig(vocab_size=32, hidden_size=24, num_hidden_layers=1,
                     num_attention_heads=4, intermediate_size=48, num_labels=9)
for kind in ["cls","mean","label_attention"]:
    tiny.head_type = kind
    m = SymptomModel(tiny).eval()
    with torch.no_grad():
        v = m(torch.tensor([[0,5,6,2,1],[0,2,1,1,1]]),
              torch.tensor([[1,1,1,1,0],[1,1,0,0,0]]), torch.zeros(2,9))
    assert v.logits.shape == (2,9) and torch.isfinite(v.loss)
del m
print("head 구조 검사 OK")

# Real save/load regression test using a tiny local model (no download).
import tempfile
def verify_loss(model, batch, expected_weights):
    model.eval()
    with torch.no_grad():
        result = model(**batch)
        z, y = result.logits.float(), batch["labels"].float()
        w = torch.tensor(expected_weights, dtype=torch.float32, device=z.device)
        # Independent BCE identity, with no reference to model.config weights.
        manual = ((1-y)*nn.functional.softplus(z) + w*y*nn.functional.softplus(-z)).mean()
    torch.testing.assert_close(result.loss, manual, rtol=1e-5, atol=1e-6)
    assert model.config.loss_pos_weight == list(expected_weights)
    return result

for test_weights in [[1.]*9, [2.]*9]:
    tiny.head_type = "label_attention"
    tiny.loss_pos_weight = test_weights
    test_model = SymptomModel(tiny).eval()
    test_batch = {"input_ids": torch.tensor([[0,5,6,2],[0,7,8,2]]),
                  "attention_mask": torch.ones(2,4,dtype=torch.long),
                  "labels": torch.tensor([[1,0,1,0,1,0,1,0,1],[0,1,0,1,0,1,0,1,0]],dtype=torch.float32)}
    before = verify_loss(test_model, test_batch, test_weights)
    with tempfile.TemporaryDirectory() as test_dir:
        test_model.save_pretrained(test_dir)
        loaded = SymptomModel.from_pretrained(test_dir).eval()
        after = verify_loss(loaded, test_batch, test_weights)
        torch.testing.assert_close(before.logits, after.logits)
    del test_model, loaded
print("BCE independent calculation + save/load checks passed")

from transformers import TrainerCallback
class FiniteGradientCheck(TrainerCallback):
    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        for name, parameter in model.named_parameters():
            if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
                raise FloatingPointError(f"Non-finite gradient: {name}")
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and metrics.get("eval_macro_f1") == 0:
            raise RuntimeError("Epoch evaluation F1 is zero. Stop and inspect; do not run L1 yet.")


## 6. 학습 함수
기존 결과는 덮어쓰지 않습니다. 남아 있는 checkpoint가 있으면 이어서 학습합니다. B0/L1 초기 파라미터 해시를 비교합니다.


In [ ]:
def run_experiment(experiment):
    mode = "boundary"
    assert experiment in EXPERIMENTS
    kind = "label_attention"
    out = OUT / experiment
    if (out / "metrics.json").exists():
        print("완료된 실험입니다. 저장 결과:", json.loads((out / "metrics.json").read_text()))
        return
    out.mkdir(exist_ok=True)
    checkpoints = sorted((out / "checkpoints").glob("checkpoint-*"),
                         key=lambda p: int(p.name.split("-")[-1]))
    resume = str(checkpoints[-1]) if checkpoints else None
    set_seed(42)
    config = RobertaConfig.from_pretrained(MODEL_NAME)
    config.num_labels = 9
    config.id2label = dict(enumerate(TARGETS))
    config.label2id = {t:i for i,t in enumerate(TARGETS)}
    config.head_type = kind
    config.loss_pos_weight = POS_WEIGHTS.tolist() if experiment == "l1_weighted_bce" else [1.] * 9
    model = SymptomModel.from_pretrained(MODEL_NAME, config=config)
    expected_weights = POS_WEIGHTS.tolist() if experiment == "l1_weighted_bce" else [1.] * 9
    # Use real examples containing every target, so broken positive loss cannot hide.
    checks = sorted(set(int(np.flatnonzero(train[TARGETS].to_numpy()[:, j])[0]) for j in range(9)))
    batch = collate([train_ds[i] for i in checks])
    assert (batch["labels"].sum(0) > 0).all()
    initial = verify_loss(model, batch, expected_weights)
    assert initial.logits.abs().max() < 20, "Extreme initial logits; stop before training"
    print("Loss weights:", expected_weights)
    print("Initial BCE verified:", initial.loss.item(), "positive labels:", batch["labels"].sum(0).tolist())
    model.train()
    # 가중치 buffer를 제외한 학습 파라미터는 두 조건이 같아야 합니다.
    initial_digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        initial_digest.update(name.encode())
        initial_digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    initial_hash = initial_digest.hexdigest()
    initial_path = OUT / "initial_parameter_sha256.txt"
    if initial_path.exists():
        assert initial_path.read_text() == initial_hash, "초기 가중치가 다릅니다. 환경/모델 revision을 확인하세요."
    else:
        initial_path.write_text(initial_hash)
    kwargs = dict(output_dir=str(out / "checkpoints"), num_train_epochs=4,
                  per_device_train_batch_size=8, per_device_eval_batch_size=16,
                  gradient_accumulation_steps=1, learning_rate=2e-5, warmup_steps=1000,
                  weight_decay=.01, logging_steps=50, logging_first_step=True, fp16=False, bf16=False, tf32=False, max_grad_norm=1.0, report_to="none",
                  seed=42, data_seed=42, save_strategy="epoch", save_total_limit=1,
                  load_best_model_at_end=False)
    key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
    kwargs[key] = "epoch"
    provenance = {"implementation": "loss_config_fp32_v2", "head": kind, "model": MODEL_NAME, "model_revision": getattr(config,"_commit_hash",None),
                  "threshold": .5, "loss": experiment, "pos_weight": config.loss_pos_weight,
                  "initial_parameter_sha256": initial_hash, "selection": "final_epoch",
                  "input": "first_512", "preprocessing": mode,
                  "normalization": "NFC + whitespace" if mode == "boundary_normalized" else "none", "targets": TARGETS, "environment": ENV,
                  "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
                  "training": kwargs}
    previous = out / "run_config.json"
    if previous.exists():
        assert json.loads(previous.read_text()) == provenance, "기존 설정과 다릅니다. OUT 경로를 변경하세요."
    previous.write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
    trainer = Trainer(model=model, args=TrainingArguments(**kwargs), train_dataset=train_ds,
                      eval_dataset=valid_ds, data_collator=collate, compute_metrics=compute_metrics,
                      callbacks=[FiniteGradientCheck()])
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(out / "model"))
    tokenizer.save_pretrained(out / "model")
    logits = trainer.predict(valid_ds).predictions
    if isinstance(logits, tuple): logits = logits[0]
    probs = torch.sigmoid(torch.as_tensor(logits).float()).numpy()
    y = valid[TARGETS].to_numpy(dtype=int)
    np.savez_compressed(out / "dev_predictions.npz", probs=probs, y_true=y,
                        file_names=valid.file_name.to_numpy(dtype=str), targets=np.asarray(TARGETS))
    assert probs.shape == y.shape and np.isfinite(probs).all()
    report = label_report(y, probs)
    report.to_csv(out / "per_label.csv", index=False)
    result = {"fixed_threshold": .5,
              "fixed_threshold_macro_f1": float(f1_score(y, probs >= .5, average="macro", zero_division=0)),
              "head": kind, "loss": experiment, "pos_weight": config.loss_pos_weight, "preprocessing": mode, "evaluation": "internal validation", "selection": "final_epoch"}
    (out / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))
    (out / "metrics.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
    print(result)
    display(report.sort_values("f1"))
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()


## 6-1. 고정 TF-IDF 학습 (한 번만)
입력은 공백 연결 원문 전체입니다. 내부 Training에만 fit합니다. 두 손실 조건에서 같은 예측을 재사용합니다.
모델 joblib에는 학습된 어휘가 포함될 수 있으므로 Git에 올리지 마세요.


In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

TF_DIR = OUT / "tfidf_fixed"
TF_DIR.mkdir(exist_ok=True)
tf_config = {"input": "raw full text", "fit_split": "train only", "threshold": .5,
             "analyzer": "char", "ngram_range": [2,5], "min_df": 2,
             "max_features": 150000, "sublinear_tf": True, "C": 2.,
             "max_iter": 1000, "solver": "liblinear", "seed": 42,
             "environment": ENV,
             "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest()}
config_path = TF_DIR / "run_config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == tf_config, "설정이 다릅니다. OUT을 변경하세요."
config_path.write_text(json.dumps(tf_config, ensure_ascii=False, indent=2))

def check_probs(probs):
    assert probs.shape == (len(valid), len(TARGETS))
    assert np.isfinite(probs).all() and ((probs >= 0) & (probs <= 1)).all()

def export_result(folder, probs, extra):
    folder.mkdir(parents=True, exist_ok=True)
    check_probs(probs)
    truth = valid[TARGETS].to_numpy(dtype=int)
    np.savez_compressed(folder / "dev_predictions.npz", probs=probs, y_true=truth,
                        file_names=valid.file_name.to_numpy(dtype=str), targets=np.asarray(TARGETS))
    label_report(truth, probs).to_csv(folder / "per_label.csv", index=False)
    result = {"fixed_threshold": .5,
              "fixed_threshold_macro_f1": float(f1_score(truth, probs >= .5, average="macro", zero_division=0)),
              "evaluation": "internal validation", **extra}
    (folder / "metrics.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
    return result

if (TF_DIR / "dev_predictions.npz").exists():
    with np.load(TF_DIR / "dev_predictions.npz", allow_pickle=False) as a:
        assert np.array_equal(a["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(a["y_true"], valid[TARGETS].to_numpy())
        assert list(a["targets"]) == TARGETS
        tf_probs = a["probs"].copy()
else:
    tf_model = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2,5), min_df=2,
                                 max_features=150000, sublinear_tf=True, dtype=np.float32)),
        ("classifier", OneVsRestClassifier(LogisticRegression(C=2., max_iter=1000,
                                       solver="liblinear", random_state=42), n_jobs=1))])
    tf_model.fit(train.text, train[TARGETS])
    tf_probs = tf_model.predict_proba(valid.text)
    joblib.dump(tf_model, TF_DIR / "model.joblib")
    del tf_model
    gc.collect()
print(export_result(TF_DIR, tf_probs, {"model": "TF-IDF + LR", "input": "raw full text"}))

RO_BERTA_WEIGHT = .5  # 결과를 보고 탐색하지 않습니다.
def evaluate_blend(mode):
    p = OUT / mode / "dev_predictions.npz"
    with np.load(p, allow_pickle=False) as a:
        assert np.array_equal(a["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(a["y_true"], valid[TARGETS].to_numpy())
        assert list(a["targets"]) == TARGETS
        ro_probs = a["probs"].copy()
    check_probs(ro_probs)
    check_probs(tf_probs)
    result = export_result(OUT / (mode + "_ensemble"),
                           RO_BERTA_WEIGHT * ro_probs + (1-RO_BERTA_WEIGHT) * tf_probs,
                           {"model": "KLUE-RoBERTa + TF-IDF", "roberta_head": "label_attention",
                            "experiment": mode, "roberta_preprocessing": "boundary", "tfidf_preprocessing": "raw full text",
                            "roberta_weight": RO_BERTA_WEIGHT, "tfidf_weight": 1-RO_BERTA_WEIGHT})
    print("앙상블:", result)


## 7. B0 — 일반 BCE 대조군
학습 후 11번 결과 백업 셀을 실행해 중간 결과도 보관할 수 있습니다.


In [ ]:
run_experiment("b0_bce")
evaluate_blend("b0_bce")


## 8. L1 — 가중 BCE
동일 입력·동일 초기 가중치로 시작하고 손실함수만 변경합니다.


In [ ]:
run_experiment("l1_weighted_bce")
evaluate_blend("l1_weighted_bce")


## 9. 비교 전 검증
예측의 통화 순서와 정답이 같고 두 학습의 초기 모델 파라미터가 같은지 확인합니다.


In [ ]:
configs = [json.loads((OUT / e / "run_config.json").read_text()) for e in EXPERIMENTS]
assert configs[0]["initial_parameter_sha256"] == configs[1]["initial_parameter_sha256"]
args0, args1 = [dict(c["training"]) for c in configs]
args0.pop("output_dir"); args1.pop("output_dir")
assert args0 == args1
for e in EXPERIMENTS:
    with np.load(OUT / e / "dev_predictions.npz", allow_pickle=False) as a:
        assert np.array_equal(a["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(a["y_true"], valid[TARGETS].to_numpy())
print("비교 조건 검증 완료")


## 10. 결과 비교
전체 Macro F1과 9개 증상의 precision/recall/F1, TP/FP/FN을 확인합니다. recall 증가만으로 성공이라고 판단하지 않습니다.


In [ ]:
summary = []
for name in ["tfidf_fixed"] + [x for mode in EXPERIMENTS for x in [mode, mode + "_ensemble"]]:
    folder = OUT / name
    if not (folder / "metrics.json").exists():
        continue
    result = json.loads((folder / "metrics.json").read_text())
    with np.load(folder / "dev_predictions.npz", allow_pickle=False) as a:
        assert np.array_equal(a["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(a["y_true"], valid[TARGETS].to_numpy())
        check_probs(a["probs"])
        label = label_report(a["y_true"], a["probs"]).set_index("symptom").loc["오심"]
    summary.append({"experiment": name, "macro_f1": result["fixed_threshold_macro_f1"],
                    "nausea_precision": label.precision, "nausea_recall": label.recall,
                    "nausea_f1": label.f1})
summary = pd.DataFrame(summary)
display(summary)
summary.to_csv(OUT / "summary.csv", index=False)
print("b0_bce와 l1_weighted_bce가 주 비교입니다. _ensemble은 고정 TF-IDF 결합 결과입니다.")
details = []
for experiment in EXPERIMENTS:
    with np.load(OUT / experiment / "dev_predictions.npz", allow_pickle=False) as a:
        y = a["y_true"].astype(bool); pred = a["probs"] >= .5
        r = label_report(y, a["probs"])
        r["experiment"] = experiment
        r["TP"] = (y & pred).sum(0)
        r["FP"] = (~y & pred).sum(0)
        r["FN"] = (y & ~pred).sum(0)
        details.append(r)
details = pd.concat(details, ignore_index=True)
details.to_csv(OUT / "label_comparison.csv", index=False)
display(details)
scores = summary.set_index("experiment").macro_f1
print("가중 BCE - BCE:", float(scores["l1_weighted_bce"] - scores["b0_bce"]))
print("단일 seed의 내부 검증 결과입니다. 개선 여부 확인 후에만 다음 문맥 실험을 진행하세요.")


## 11. 매 실험 후 실행 가능한 작은 결과 백업
예측·지표·분할·로그·실험 설정을 포함합니다. 학습 가중치와 원문 오류 CSV는 제외합니다. 노트북 코드는 별도로 저장하세요.
이 백업으로 내부 성능 재계산은 가능하지만 새 데이터 추론·학습 재개에는 모델/checkpoint 백업도 필요합니다.
노트북 자체도 파일 → 다운로드 → .ipynb로 저장하세요.
TF-IDF model.joblib도 작은 결과 백업에서 제외됩니다. 학습된 어휘·가중치 복원에는 모델 전체 백업을 사용하세요.


In [ ]:
archive_path = Path("/content/m3_loss_control_fixed_v2_results.tar.gz")
with tarfile.open(archive_path, "w:gz") as a:
    for p in OUT.rglob("*"):
        if p.is_file() and "model" not in p.relative_to(OUT).parts and "checkpoints" not in p.relative_to(OUT).parts:
            if p.name != "e0_nausea_errors_private.csv" and p.suffix != ".joblib":
                a.add(p, arcname=str(p.relative_to(OUT)))
print("SHA256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())
files.download(str(archive_path))


## 12. 선택: 모델까지 Drive 백업
Drive 여유 공간을 먼저 확인하세요. 모델은 크며 작은 결과 백업에는 없습니다.
아래 BACKUP_MODELS를 True로 바꿔야 복사합니다. checkpoint를 포함하므로 용량이 수 GB 이상 필요할 수 있습니다.
새 경로만 허용하고 기존 백업은 덮어쓰지 않습니다.


In [ ]:
BACKUP_MODELS = False
if BACKUP_MODELS:
    import shutil
    from datetime import datetime
    destination = Path("/content/drive/MyDrive/DDC_M3/backups") / ("loss_control_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT, destination)
    print("모델·체크포인트 포함 백업:", destination)
else:
    print("모델 백업을 건너뜁니다. 결과 압축파일에는 모델이 없습니다.")


복원 예시 (모델 클래스 정의 셀을 먼저 실행):
`SymptomModel.from_pretrained("/content/.../model")`

설계 참고: [Hugging Face Trainer](https://huggingface.co/docs/transformers/main_classes/trainer),
[Label-Specific Attention 연구](https://aclanthology.org/D19-1044/).
이 노트북은 해당 논문 전체 재현이 아닌 간단한 증상별 attention 비교 구현입니다.
